In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA01 - Travel, Lodging, and Entertainment
   
   BUSINESS QUESTION: 
   During the assessment period, did the Unit provide or receive gifts or anything of 
   value to/from OR pay for the travel, hospitality or entertainment of Customers 
   and Third Parties?
   
   LOGIC SUMMARY:
   1. TARGET CATEGORY CODES: Grabs the full list of Category Codes from the ABAC 
      category_codes_database. Any code in this table is considered flagged.
   2. COUPA DATA (7 Files): Unions the 7 Coupa files. Parses the 'Account' string 
      (format: xxxx-xxxx-xxxxxx) to extract the Cost Center (first block) and Category 
      Code (last block). Uses TRY_CAST to INT to safely strip leading zeros.
   3. FINANCE DATA (4 Files verified): Unions the Finance files. Extracts the Cost Center 
      (casting to INT) and Category Code directly from their respective columns.
   4. CONSOLIDATION & FILTERING: Combines Coupa and Finance data into one master list 
      of transactions. INNER JOINS to the ABAC category_codes_database to strictly keep 
      transactions that match a flagged Category Code.
   5. MAPPING & OUTPUT: Joins the standardized Cost Center Mapping Bootstrap to the 
      flagged transactions. Rolls up by AU_ID. If any flagged transaction exists for 
      the Assessable Unit, outputs 'Yes'. Otherwise, 'No'.
=================================================================================== */

WITH Target_Category_Codes AS (
    -- 1. Grab the ABAC category codes (Any transaction matching these is a "Yes")
    SELECT DISTINCT TRIM(CatCode) AS CatCode
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 2a. Union all 7 Coupa files into one master dataset
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 2b. Parse the Coupa Account string: splits by '-' 
    -- [0] is the first part (CC), [2] is the third part (Category Code)
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        'Coupa' AS Source_System
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%' -- Ensures the string actually has the correct hyphen format
),

Combined_Finance_Raw AS (
    -- 3a. Union all available Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    -- Uncomment below if files 5 and 6 exist!
    -- UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    -- UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    -- 3b. Clean the Finance columns
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        'Finance' AS Source_System
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    -- 4a. Stack Coupa and Finance data together
    SELECT Cleaned_CC, CatCode, Source_System FROM Coupa_Parsed
    UNION ALL
    SELECT Cleaned_CC, CatCode, Source_System FROM Finance_Parsed
),

Flagged_Transactions AS (
    -- 4b. Strictly keep transactions where the Category Code matches the ABAC database
    SELECT 
        t.Cleaned_CC,
        t.CatCode,
        t.Source_System
    FROM All_Transactions t
    INNER JOIN Target_Category_Codes c
        ON t.CatCode = c.CatCode
    WHERE t.Cleaned_CC IS NOT NULL
),

Mapped_AUs AS (
    -- 5a. Join the flagged transactions to the standardized CC Mapping
    SELECT 
        m.AU_ID AS BusinessID,
        m.AU_Name,
        CASE WHEN f.Cleaned_CC IS NOT NULL THEN 1 ELSE 0 END AS Has_Flagged_Transaction
    FROM vw_cost_center_mapping_bootstrap m
    
    -- LEFT JOIN to flagged data so all AUs appear in the final output
    LEFT JOIN Flagged_Transactions f
        ON CAST(m.Cost_Center_ID AS INT) = f.Cleaned_CC
)

-- 5b. Roll everything up to match the Final Output Template
SELECT 
    BusinessID,                          
    AU_Name,                             
    'EBA01' AS QuestionID,               
    'Applicable' AS Applicability,       
    '' AS ApplicabilityRationale,        
    
    CASE 
        WHEN SUM(Has_Flagged_Transaction) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response
    
FROM Mapped_AUs
GROUP BY BusinessID, AU_Name
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA01 - Flagged Transactions Drill-Down
=================================================================================== */

WITH Target_Category_Codes AS (
    SELECT DISTINCT TRIM(CatCode) AS CatCode, Description, Purpose
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

All_Raw_Transactions AS (
    -- Stack Coupa
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        'Coupa' AS Source_System,
        Account AS Raw_System_String
    FROM (
        SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
        UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
        UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
        UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
        UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
        UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
        UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
    ) WHERE Account LIKE '%-%-%'
    
    UNION ALL
    
    -- Stack Finance
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        'Finance' AS Source_System,
        CatCode AS Raw_System_String
    FROM (
        SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
        UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
        UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
        UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    )
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    t.Source_System,
    t.CatCode AS Extracted_Category_Code,
    c.Description AS ABAC_Category_Description,
    t.Raw_System_String
    
FROM vw_cost_center_mapping_bootstrap m

-- INNER JOIN to only return rows that actually had a mapped transaction
INNER JOIN All_Raw_Transactions t
    ON CAST(m.Cost_Center_ID AS INT) = t.Cleaned_CC
    
-- INNER JOIN to only keep transactions that matched the ABAC risk list
INNER JOIN Target_Category_Codes c
    ON t.CatCode = c.CatCode
    
-- FIX: Changed m.AU_ID to BusinessID
ORDER BY BusinessID, Source_System;